# Predicting Fish Weight from Body Measurements

**A comparison of linear and non-linear regression models on the Fish Market dataset.**

---

## Problem

Given physical measurements of a fish (species, three length measurements, height, and width), can we accurately predict its weight? This is a classic regression problem with an interesting twist: from basic physics, weight should scale with volume, i.e. approximately with the *cube* of length. That gives us a strong prior about model form and lets us test whether classic linear regression is even the right family for this task.

## Dataset

- **Source:** [Fish Market dataset on Kaggle](https://www.kaggle.com/datasets/aungpyaeap/fish-market)
- **Size:** 159 samples, 7 columns
- **Target:** `Weight` (grams)
- **Features:** `Species` (categorical), `Length1`, `Length2`, `Length3`, `Height`, `Width` (all in cm)

## Approach

1. Explore the data and the target distribution.
2. Investigate the physical hypothesis that weight ~ length³.
3. Establish two baselines: a mean predictor and a univariate linear regression (weight ~ Length1 only).
4. Fit and cross-validate five candidate models: multivariate linear, Ridge, Lasso, polynomial features + Ridge, and Random Forest.
5. Perform residual diagnostics on the best model.
6. Report feature importance and discuss limitations.

## How to run

Upload `Fish.csv` to your Colab environment (or update the `DATA_PATH` variable below). The notebook is reproducible — all random seeds are fixed.

---

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Style
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 100

DATA_PATH = '/content/Fish.csv'

## 2. Load and inspect the data

In [ ]:
data = pd.read_csv(DATA_PATH)
print(f'Shape: {data.shape}')
data.head()

In [ ]:
data.info()

In [ ]:
# Sanity checks: any missing values? any impossible zeros?
print('Missing values per column:')
print(data.isnull().sum())
print()
print(f'Rows where Weight == 0: {(data["Weight"] == 0).sum()}')

**TODO (interpretation):** If any rows have `Weight == 0`, decide how to handle them (drop them? These are likely data-entry errors). Write your reasoning here.

In [ ]:
# Drop any rows with weight = 0 (impossible in practice)
data = data[data['Weight'] > 0].reset_index(drop=True)
print(f'Shape after cleaning: {data.shape}')

## 3. Exploratory data analysis

### 3.1 Target distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data['Weight'], kde=True, ax=axes[0])
axes[0].set_title('Weight distribution (raw)')
axes[0].set_xlabel('Weight (g)')

sns.histplot(np.log(data['Weight']), kde=True, ax=axes[1], color='C1')
axes[1].set_title('Weight distribution (log-transformed)')
axes[1].set_xlabel('log(Weight)')
plt.tight_layout()
plt.show()

**TODO (interpretation):** Which is more symmetric — the raw or log-transformed weight? Right-skewed targets often benefit from log-transformation before fitting a linear model.

### 3.2 Correlations

In [ ]:
numeric_cols = data.select_dtypes(include=np.number).columns
plt.figure(figsize=(8, 6))
sns.heatmap(data[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation matrix')
plt.show()

**TODO (interpretation):** Which features correlate most strongly with weight? Do the length features correlate strongly with each other? (Multicollinearity is why we will test Ridge and Lasso later.)

### 3.3 The physical prior: weight ~ length³

An object's mass equals its volume times its density. Volume scales with length³, so we expect `Weight ≈ k * Length³` for constant-density fish. A straight-line fit of `Weight` on `Length1` will systematically under-predict for large fish and over-predict for small ones.

In [ ]:
# Compare linear fit vs cubic fit vs log-log
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

x = data['Length1'].values
y = data['Weight'].values

# Linear scale
axes[0].scatter(x, y, alpha=0.6)
axes[0].set_xlabel('Length1 (cm)')
axes[0].set_ylabel('Weight (g)')
axes[0].set_title('Weight vs Length1 (linear)')

# Cubic curve overlaid
x_line = np.linspace(x.min(), x.max(), 100)
k = np.mean(y / x**3)
axes[1].scatter(x, y, alpha=0.6)
axes[1].plot(x_line, k * x_line**3, color='red', label=f'y = {k:.4f} x^3')
axes[1].set_xlabel('Length1 (cm)')
axes[1].set_ylabel('Weight (g)')
axes[1].set_title('Weight vs Length1 with cubic curve')
axes[1].legend()

# Log-log scale — a power law appears as a straight line
axes[2].scatter(np.log(x), np.log(y), alpha=0.6)
slope, intercept = np.polyfit(np.log(x), np.log(y), 1)
axes[2].plot(np.log(x_line), slope * np.log(x_line) + intercept,
             color='red', label=f'slope = {slope:.2f}')
axes[2].set_xlabel('log(Length1)')
axes[2].set_ylabel('log(Weight)')
axes[2].set_title('log-log plot (slope ≈ 3 confirms cubic scaling)')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f'Fitted log-log slope: {slope:.3f}  (theory predicts ~3)')

**TODO (interpretation):** How close is the fitted log-log slope to 3? Discuss what this means for the choice of model family. This is a real domain insight — mention it in your README.

## 4. Preprocessing

We one-hot encode `Species` and separate features from the target. We use ALL available features, not just `Length1`.

In [ ]:
# One-hot encode species
data_encoded = pd.get_dummies(data, columns=['Species'], drop_first=True, dtype=int)
print(f'Shape after encoding: {data_encoded.shape}')
print(f'Columns: {list(data_encoded.columns)}')

In [ ]:
X = data_encoded.drop('Weight', axis=1)
y = data_encoded['Weight']

print(f'Features shape: {X.shape}')
print(f'Target shape:   {y.shape}')

## 5. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')

## 6. Evaluation helper

To keep results consistent, we define a single function that reports R², RMSE, and MAE on the test set.

In [ ]:
def evaluate(model, X_train, y_train, X_test, y_test, name='Model'):
    """Fit the model, predict on the test set, and return a dict of metrics."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    metrics = {
        'model': name,
        'r2':   r2_score(y_test, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
        'mae':  mean_absolute_error(y_test, y_pred),
    }
    return metrics, y_pred

results = []

## 7. Baselines

Any model we build must beat these. If it doesn't, we have a bug or an overfitting problem.

### 7.1 Mean baseline

Always predict the training-set mean, ignoring all features.

In [ ]:
from sklearn.dummy import DummyRegressor
mean_model = DummyRegressor(strategy='mean')
m, _ = evaluate(mean_model, X_train, y_train, X_test, y_test, 'Mean baseline')
results.append(m)
print(m)

### 7.2 Original univariate baseline (Weight ~ Length1)

This is the model from the original notebook — we keep it as a benchmark to show what adding features and better models buys us.

In [ ]:
X_train_uni = X_train[['Length1']]
X_test_uni  = X_test[['Length1']]

uni_model = LinearRegression()
m, _ = evaluate(uni_model, X_train_uni, y_train, X_test_uni, y_test,
                'Linear (Length1 only)')
results.append(m)
print(m)

## 8. Multivariate models

In [ ]:
# All features, plain linear regression
m, _ = evaluate(LinearRegression(), X_train, y_train, X_test, y_test,
                'Linear (all features)')
results.append(m)
print(m)

In [ ]:
# Ridge — L2 regularization, helpful when features are correlated
ridge = Pipeline([('scaler', StandardScaler()),
                  ('ridge', Ridge(alpha=1.0, random_state=SEED))])
m, _ = evaluate(ridge, X_train, y_train, X_test, y_test, 'Ridge')
results.append(m)
print(m)

In [ ]:
# Lasso — L1 regularization, performs feature selection
lasso = Pipeline([('scaler', StandardScaler()),
                  ('lasso', Lasso(alpha=1.0, random_state=SEED, max_iter=10000))])
m, _ = evaluate(lasso, X_train, y_train, X_test, y_test, 'Lasso')
results.append(m)
print(m)

In [ ]:
# Polynomial features + Ridge — motivated by the physical prior
# (cubic scaling means degree-3 polynomial features should help)
poly_ridge = Pipeline([
    ('poly',   PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('ridge',  Ridge(alpha=1.0, random_state=SEED)),
])
m, poly_pred = evaluate(poly_ridge, X_train, y_train, X_test, y_test,
                        'Polynomial (deg=2) + Ridge')
results.append(m)
print(m)

In [ ]:
# Random Forest — captures non-linearities automatically
rf = RandomForestRegressor(n_estimators=200, random_state=SEED)
m, rf_pred = evaluate(rf, X_train, y_train, X_test, y_test, 'Random Forest')
results.append(m)
print(m)

## 9. Results comparison

In [ ]:
results_df = pd.DataFrame(results).set_index('model')
results_df = results_df.sort_values('r2', ascending=False)
print(results_df.round(3))

# Plot RMSE comparison
fig, ax = plt.subplots(figsize=(10, 5))
results_df['rmse'].plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('RMSE (grams)')
ax.set_title('Model comparison — lower is better')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**TODO (interpretation):** Which model won? By how much did it beat the univariate baseline? Was the improvement worth the extra complexity? Comment on what this tells you about the problem.

## 10. Cross-validated confidence

A single train/test split can be lucky. Here we run 5-fold cross-validation on the top model to check that our results are stable.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(rf, X, y, cv=cv, scoring='r2')

print(f'5-fold CV R² scores: {np.round(cv_scores, 3)}')
print(f'Mean: {cv_scores.mean():.3f}  |  Std: {cv_scores.std():.3f}')

## 11. Residual diagnostics for the best model

Even a high R² can hide systematic errors. Residuals should look like random noise around zero. Patterns in the residuals mean the model is missing something.

In [ ]:
# Refit best model and get predictions
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Predicted vs actual
axes[0].scatter(y_test, y_pred, alpha=0.7)
lo, hi = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0].plot([lo, hi], [lo, hi], 'r--', label='Perfect')
axes[0].set_xlabel('Actual weight (g)')
axes[0].set_ylabel('Predicted weight (g)')
axes[0].set_title('Predicted vs actual')
axes[0].legend()

# Residuals vs predicted
axes[1].scatter(y_pred, residuals, alpha=0.7)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted weight (g)')
axes[1].set_ylabel('Residual (actual - predicted)')
axes[1].set_title('Residuals vs predicted')

# Residual distribution
sns.histplot(residuals, kde=True, ax=axes[2])
axes[2].axvline(0, color='red', linestyle='--')
axes[2].set_xlabel('Residual (g)')
axes[2].set_title('Residual distribution')

plt.tight_layout()
plt.show()

**TODO (interpretation):** Are the residuals centered on zero? Do you see any pattern (e.g. bigger errors for bigger fish — which would suggest heteroscedasticity)? Would log-transforming the target improve this?

## 12. Feature importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values()

plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='seagreen')
plt.title('Random Forest — feature importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print(importances.sort_values(ascending=False).round(3))

**TODO (interpretation):** Which features does the model actually rely on? Are the species indicators helping (i.e. do different species have systematically different weight-to-length ratios)?

## 13. Limitations and future work

**TODO — write this yourself.** For a genuinely strong project, discuss honestly:

- Sample size is small (~159 rows), so results have real uncertainty. What are the CV standard deviations telling you?
- Hyperparameters were not tuned. Try `GridSearchCV` over Ridge `alpha`, Random Forest `max_depth` and `n_estimators`.
- The target could be log-transformed to make errors homoscedastic.
- The physical prior (weight ~ length³) could be built into the model directly — e.g. by adding `Length1**3` as an engineered feature, or fitting the power law explicitly with `np.polyfit(np.log(x), np.log(y), 1)`.
- Domain generalization: this model is trained on 7 species. It will not extrapolate to unseen species.

---

## References

1. Fish Market dataset — Aung Pyae, Kaggle. [https://www.kaggle.com/datasets/aungpyaeap/fish-market](https://www.kaggle.com/datasets/aungpyaeap/fish-market)
2. Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* — chapters 3 (linear methods) and 15 (random forests).

---

*Author: [Your Name] · [Your GitHub] · [Your Email]*